In [ ]:
from imblearn.pipeline import Pipeline
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import (
    KBinsDiscretizer,
    OneHotEncoder,
    PowerTransformer,
)

from imblearn.over_sampling import SMOTE, SMOTENC

In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [ ]:
cat_idx = [i for i, dt in enumerate(X_train.dtypes) if dt == "str"]
print(cat_idx)

In [ ]:
sm = SMOTE(random_state=42, k_neighbors=5)
smc = SMOTENC(random_state=42, k_neighbors=5, categorical_features=cat_idx)

ct = ColumnTransformer(
    [
        (
            "Categorical columns",
            OneHotEncoder(handle_unknown="ignore"),
            make_column_selector(dtype_include="str"),
        ),
        (
            "Age",
            KBinsDiscretizer(n_bins=5, encode="onehot", strategy="quantile"),
            ["age"],
        ),
        ("Balance", PowerTransformer(method="yeo-johnson"), ["balance"]),
    ],
    remainder="passthrough",
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
log_reg_pipe = Pipeline(
    [("preprocessor", ct), ("model", LogisticRegression(max_iter=1000))]
)
log_reg_balanced_pipe = Pipeline(
    [
        ("preprocessor", ct),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)
log_reg_smote_pipe = Pipeline(
    [("preprocessor", ct), ("SMOTE", sm), ("model", LogisticRegression(max_iter=1000))]
)
log_reg_smotenc_pipe = Pipeline(
    [
        ("SMOTENC", smc),
        ("preprocessor", ct),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

pipes = [log_reg_pipe, log_reg_balanced_pipe, log_reg_smote_pipe, log_reg_smotenc_pipe]

In [ ]:
for pipe in pipes:
    log_cv = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="average_precision")
    mean_ap = np.mean(log_cv)
    print(
        f"mean of logistic regression: {mean_ap}, standard deviation: {np.std(log_cv)}"
    )

We can see that synthetic data in SMOTE and SMOTENC does not upgrade model performance and the best performance on average precision show default Logistic Regression. However i want to see what probabilities give default and class weight = balanced Logistic Regression

In [ ]:
log_reg_pipe.fit(X_train, y_train)
log_reg_balanced_pipe.fit(X_train, y_train)
d_proba = log_reg_pipe.predict_proba(X_train)[:, 1]
b_proba = log_reg_balanced_pipe.predict_proba(X_train)[:, 1]

df = pd.DataFrame(
    {
        "y_true": y_train.values,
        "default_proba": d_proba,
        "balanced_proba": b_proba,
    }
)

print(df.head(20))
print(df.describe())

We can see that default logistic Regression predict probability of 0.11 and balanced of 0.41. Thus we will use Balanced Logistic Regression because we use threshold not just ranging. But the most important step is to choose the best threshold otherwise our model will perform badly.